# Schema Validation

1. What is Schema-on-read / Schema-on-write? 
2. Column order validation
3. Data Type validation
4. Column name validation
5. Nullability validation
6. Extra columns validation

In [ ]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("deltalake-schema").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1b34fc3e-ae16-45ac-8819-a18a6adc85f5;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 162ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [default

In [1]:
%load_ext sql

In [4]:
%sql spark

In [ ]:
df = spark.read.format("parquet").load("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df.printSchema()

#
display(df)

# Save as delta table into Minio S3
df.write.format('delta').save('/deltalake/invoices_sv')

root
 |-- customer_id: integer (nullable = true)
 |-- invoice_no: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- shopping_mall: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

+-----------+----------+------+---+---------------+--------+-------+--------------+------------+----------------+-------------+
|customer_id|invoice_no|gender|age|category       |quantity|price  |payment_method|invoice_date|shopping_mall   |_rescued_data|
+-----------+----------+------+---+---------------+--------+-------+--------------+------------+----------------+-------------+
|1          |I178410   |Male  |61 |Clothing       |5       |1500.4 |Credit Card   |2021-11-26  |Metrocity       |NULL         |
|2          |I158163   |Ma

In [17]:
%%sql

DROP TABLE IF EXISTS delta.`/deltalake/invoices_sv`;

Running query in 'SparkSession'

++
||
++
++

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/invoices_sv --recursive

delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000000.crc
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000000.json
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000001.crc
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000002.crc
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000002.json
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000001.json
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000004.crc
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000003.crc
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000004.json
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000003.json
delete: s3://defaultfs/deltalake/invoices_sv/_delta_log/00000000000000000005.json
delete: s3://defaultfs/deltalake/invoices_sv/part-00000-0f8d9db0-a38e-4151-909d-e02dfb9874be-c000.snapp

##### Create Delta Table

In [ ]:
%%sql

CREATE OR REPLACE TABLE delta.`/deltalake/invoices_sv`(
  customer_id INT NOT NULL,  
  invoice_no STRING,
  quantity INT,
  price FLOAT, 
  invoice_date DATE
) USING DELTA; 



Running query in 'SparkSession'

++
||
++
++

In [22]:
%%sql


INSERT INTO delta.`/deltalake/invoices_sv`
SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`; 


Running query in 'SparkSession'

++
||
++
++

In [23]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_sv`
LIMIT 5;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
1,I178410,5,1500.4000244140625,2021-11-26
2,I158163,2,1200.3399658203125,2023-03-03
3,I262373,3,107.5199966430664,2022-12-01
4,I334895,5,26.149999618530273,2021-08-15
5,I202043,1,35.84000015258789,2021-07-25


In [24]:
%%sql

SELECT MIN(customer_id), MAX(customer_id), COUNT(*)
FROM delta.`/deltalake/invoices_sv`;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3
1,100,100


#### Scenario 1: Column Order Validation

In [25]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
SELECT quantity, invoice_no, customer_id, price, invoice_date
FROM VALUES (99999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

Running query in 'SparkSession'

++
||
++
++

In [26]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_sv`
WHERE customer_id = 99999;

Running query in 'SparkSession'

++
||
++
++

In [27]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_sv`
WHERE customer_id = 10;

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
10,I280920,3,900.239990234375,2021-04-23
10,I12345,99999,100.0,2025-01-01


In [28]:
%%sql

SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
ORDER BY customer_id DESC LIMIT 5;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
200,I856190,5,1500.4,2022-05-17
199,I205401,2,600.16,2022-03-21
198,I233680,2,71.68,2021-05-11
197,I865759,1,5.23,2021-12-04
196,I171884,3,1800.51,2021-07-25


In [29]:
%%sql

MERGE INTO delta.`/deltalake/invoices_sv` AS tgt
USING (
  SELECT quantity, invoice_no, customer_id, CAST (price AS FLOAT), invoice_date
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
  ORDER BY customer_id DESC LIMIT 5
) src 
ON tgt.customer_id = src.customer_id
WHEN MATCHED THEN 
  UPDATE SET 
    tgt.customer_id = src.customer_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.invoice_date = current_date
WHEN NOT MATCHED THEN
  INSERT * 

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4
5,0,0,5


In [30]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_sv`
WHERE customer_id > 100; 

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
200,I856190,5,1500.4000244140625,2022-05-17
199,I205401,2,600.1599731445312,2022-03-21
198,I233680,2,71.68000030517578,2021-05-11
197,I865759,1,5.230000019073486,2021-12-04
196,I171884,3,1800.510009765625,2021-07-25


#### Scenario 2: Data Type Validation

In [31]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
VALUES (
  'ABC', 
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

Running query in 'SparkSession'

NumberFormatException: [CAST_INVALID_INPUT] The value 'ABC' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error.
== SQL(line 1, position 1) ==
INSERT INTO delta.`/deltalake/invoices_sv`
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


In [32]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
VALUES (
  '99499', 
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

Running query in 'SparkSession'

++
||
++
++

#### Scenario 3: Column Name Validation

In [33]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
SELECT customer_id AS c_id, invoice_no, quantity AS qty, price, invoice_date
FROM VALUES (99999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

Running query in 'SparkSession'

++
||
++
++

In [34]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_sv`
WHERE customer_id = 99999;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
99999,I12345,10,100.0,2025-01-01


In [35]:
%%sql

SELECT customer_id, invoice_no, quantity, CAST (price AS FLOAT), invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
ORDER BY customer_id LIMIT 5

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
101,I302068,2,1200.3399658203125,2022-07-15
102,I193229,5,1500.4000244140625,2021-03-05
103,I313092,2,81.31999969482422,2022-04-23
104,I258750,3,15.6899995803833,2022-04-04
105,I126182,5,1500.4000244140625,2022-02-06


In [36]:
%%sql

MERGE INTO delta.`/deltalake/invoices_sv` AS tgt
USING (
  SELECT customer_id AS c_id, invoice_no, quantity AS qty, CAST (price AS FLOAT), invoice_date
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
  ORDER BY customer_id LIMIT 5
) src 
ON tgt.customer_id = src.c_id
WHEN MATCHED THEN 
  UPDATE SET 
    tgt.customer_id = src.c_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.qty,
    tgt.price = src.price,
    tgt.invoice_date = current_date
WHEN NOT MATCHED THEN
  INSERT * 

Running query in 'SparkSession'

RuntimeError: [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve customer_id in INSERT clause given columns src.c_id, src.invoice_no, src.qty, src.price, src.invoice_date; line 1 pos 0


In [37]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_sv`
WHERE customer_id > 100;

Running query in 'SparkSession'

7 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
200,I856190,5,1500.4000244140625,2022-05-17
199,I205401,2,600.1599731445312,2022-03-21
198,I233680,2,71.68000030517578,2021-05-11
197,I865759,1,5.230000019073486,2021-12-04
196,I171884,3,1800.510009765625,2021-07-25
99499,I45678,10,98.75,2025-01-01
99999,I12345,10,100.0,2025-01-01


#### Scenario 4: Nullability Validation

In [38]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
VALUES (NULL, NULL, NULL, NULL, NULL);

Running query in 'SparkSession'

26/03/31 14:26:31 ERROR Utils: Aborting task
org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_NOT_NULL_CONSTRAINT_VIOLATED] NOT NULL constraint violated for column: customer_id.

	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getNotNullInvariantViolationException(InvariantViolationException.scala:53)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:58)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.delta.constraints.DeltaInvariantCheckerExec.$anonfun$doExecute$3(DeltaInvariantCheckerExec.scala:89)
	at scala.collection.Iterator$$anon$10.next(Iterator.scala:461)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithIterator(FileFormatDataWriter.scala:92)
	at 

Py4JJavaError: An error occurred while calling o56.sql.
: org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_NOT_NULL_CONSTRAINT_VIOLATED] NOT NULL constraint violated for column: customer_id.

	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getNotNullInvariantViolationException(InvariantViolationException.scala:53)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:58)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.delta.constraints.DeltaInvariantCheckerExec.$anonfun$doExecute$3(DeltaInvariantCheckerExec.scala:89)
	at scala.collection.Iterator$$anon$10.next(Iterator.scala:461)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithIterator(FileFormatDataWriter.s

In [39]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
VALUES (78912, NULL, NULL, NULL, NULL);

Running query in 'SparkSession'

++
||
++
++

#### Scenario 5: Extra Column Validation

In [40]:
%%sql

INSERT INTO delta.`/deltalake/invoices_sv`
SELECT customer_id, invoice_no, quantity, price, invoice_date, "VIP" AS customer_type
FROM VALUES (78999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

Running query in 'SparkSession'

RuntimeError: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 4d7b4549-b076-4fa3-92e3-049fbac7f74e).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- customer_id: integer (nullable = false)
-- invoice_no: string (nullable = true)
-- quantity: integer (nullable = true)
-- price: float (nullable = true)
-- invoice_date: date (nullable = true)


Data schema:
root
-- customer_id: integer (nullable = true)
-- invoice_no: string (nullable = true)
-- quantity: integer (nullable = true)
-- price: float (nullable = true)
-- invoice_date: date (nullable = true)
-- customer_type: string (nullable = true)

         


In [41]:
%%sql

SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
WHERE customer_id BETWEEN 150 AND 155

Running query in 'SparkSession'

6 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
150,I500960,4,1200.32,2022-12-31
151,I125404,2,71.68,2021-05-27
152,I182701,3,3150.0,2021-02-05
153,I197462,2,1200.34,2022-06-06
154,I154490,5,1500.4,2022-01-24
155,I187525,3,900.24,2022-12-16


In [42]:
%%sql

MERGE INTO delta.`/deltalake/invoices_sv` AS tgt
USING (
  SELECT customer_id, invoice_no, quantity, price, invoice_date, "VIP" AS customer_type
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`
  WHERE customer_id BETWEEN 150 AND 155
) src 
ON tgt.customer_id = src.customer_id
WHEN NOT MATCHED THEN
  INSERT * 

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4
6,0,0,6


In [43]:
%%sql

SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM delta.`/deltalake/invoices_sv`
WHERE customer_id BETWEEN 150 AND 155

Running query in 'SparkSession'

6 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
150,I500960,4,1200.3199462890625,2022-12-31
151,I125404,2,71.68000030517578,2021-05-27
152,I182701,3,3150.0,2021-02-05
153,I197462,2,1200.3399658203125,2022-06-06
154,I154490,5,1500.4000244140625,2022-01-24
155,I187525,3,900.239990234375,2022-12-16
